<img src="https://raw.githubusercontent.com/OmarLi-360/excel-corporativo-12-claves/main/assets/logo-drbits.jpg" alt="Dr. Bits" width="170">

# Taller 3 · Tablero para el comité
**Laboratorio Excel + Colab** — Integra el mes nuevo, compara con la meta y genera el tablero web · ⏱ 25 min en Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OmarLi-360/excel-corporativo-12-claves/blob/main/notebooks/taller_3_tablero_comite.ipynb) &nbsp; [Ver el taller completo en el tablero](https://omarli-360.github.io/excel-corporativo-12-claves/#taller-3)

Este cuaderno es una parte del taller. Los pasos en Excel están en el tablero.

**Cómo usarlo:** *Archivo › Guardar una copia en Drive* → *Entorno de ejecución › Ejecutar todas*. Las celdas con listas desplegables se pueden cambiar y volver a ejecutar con ▶. No necesitas programar.

In [ ]:
# @title Preparar el entorno (se ejecuta sola con «Ejecutar todas»)
import os, sys, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, Markdown, HTML
warnings.filterwarnings("ignore")

EN_COLAB = "google.colab" in sys.modules
REPO = "OmarLi-360/excel-corporativo-12-claves"
BASE_URL = f"https://raw.githubusercontent.com/{REPO}/main/data/"
BASE_LOCAL = "../data/"

def ruta(nombre):
    local = os.path.join(BASE_LOCAL, nombre)
    return local if os.path.exists(local) else BASE_URL + nombre

# --- formatos colombianos -------------------------------------------------
def miles(x):    return f"{x:,.0f}".replace(",", ".")
def pesos(x):    return "$ " + miles(x)
def millones(x): return "$ " + f"{x/1e6:,.0f}".replace(",", ".") + " M"
def porc(v, signo=False):
    t = f"{v:+.1%}" if signo else f"{v:.1%}"
    return t.replace(".", ",").replace("%", " %")
MESES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
MM = "Millones de pesos"

# --- plantilla de gráficos Dr. Bits ----------------------------------------
AZUL, TEAL, MAGENTA, GRIS = "#4A5AD8", "#0E9BB8", "#D0336F", "#9AA3BF"
BIEN, MAL = "#15784A", "#B42318"
ejes = dict(gridcolor="#E4E8F2", zeroline=False, linecolor="#C9D0E2", exponentformat="none", separatethousands=True)
pio.templates["drbits"] = go.layout.Template(layout=dict(
    font=dict(family="Source Sans 3, Segoe UI, Arial, sans-serif", size=14, color="#11153A"),
    title=dict(font=dict(size=18, color="#00006A"), x=0, xanchor="left"),
    colorway=[AZUL, TEAL, MAGENTA], plot_bgcolor="white", paper_bgcolor="white",
    xaxis=ejes, yaxis=ejes, separators=",.", hoverlabel=dict(font_size=13),
    margin=dict(t=80, l=10, r=30, b=40)))
pio.templates.default = "plotly_white+drbits"

def eje_meses(fig, fechas, cada=None):
    """Etiquetas de meses en español en ejes de tiempo (Plotly las pone en inglés)."""
    f = pd.to_datetime(pd.Series(sorted(set(pd.to_datetime(list(fechas))))))
    paso = cada or (1 if len(f) <= 13 else 3)
    f = f.iloc[::paso]
    fig.update_xaxes(tickvals=list(f), ticktext=[f"{MESES[d.month - 1]} {d.year}" for d in f])

def dar_aire(fig, maximo, eje="x"):
    (fig.update_xaxes if eje == "x" else fig.update_yaxes)(range=[0, maximo * 1.22])

# --- revisor de retos -----------------------------------------------------
def revisar(respuesta, esperado, explicacion=""):
    if respuesta == "Elige una opción":
        display(Markdown("**Elige una opción en la lista de la derecha** y vuelve a ejecutar esta celda con ▶."))
    elif str(respuesta) == str(esperado):
        display(Markdown(f"✅ **Correcto.** {explicacion}"))
    else:
        display(Markdown(f"❌ **Todavía no.** Revisa el gráfico de arriba e inténtalo otra vez. *(Elegiste: {respuesta})*"))

# --- datos de Distribuidora Andina ----------------------------------------
ventas = pd.read_csv(ruta("ventas.csv"), parse_dates=["fecha"])
clientes = pd.read_csv(ruta("clientes.csv"))
productos = pd.read_csv(ruta("productos.csv"))
vendedores = pd.read_csv(ruta("vendedores.csv"))
sedes = pd.read_csv(ruta("sedes.csv"))
metas = pd.read_csv(ruta("metas.csv"))
df = (ventas
      .merge(sedes[["id_sede", "ciudad"]], on="id_sede", validate="many_to_one")
      .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
      .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
      .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
      .assign(anio=lambda d: d.fecha.dt.year, mes=lambda d: d.fecha.dt.month))
validas = df[df["estado"] != "Anulada"]          # las anuladas no son venta
ORDEN_CIUDADES = (validas[validas.anio == 2025].groupby("ciudad")["subtotal"].sum()
                  .sort_values(ascending=False).index.tolist())
display(Markdown(f"Datos listos: **{miles(len(ventas))} líneas de venta**, {len(clientes)} clientes, {len(productos)} productos. "
                 f"Fuente: {'archivos locales' if os.path.exists(BASE_LOCAL) else 'GitHub'}."))

> **Escenario.** Es lunes 2 de febrero de 2026. A las 9:00 se reúne el comité comercial y acaban de llegar los datos de enero. El gerente quiere saber: ¿cómo cerró enero frente a la meta?, ¿qué sede preocupa? y ¿qué está pasando con los canales?

## Paso 1 · Integrar enero de 2026

In [ ]:
# @title Cargar 2024, 2025 y enero de 2026

MESES_ES = ["Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic"]
def tabla_plana(v):
    d = (v.merge(sedes[["id_sede", "ciudad", "region"]], on="id_sede", validate="many_to_one")
          .merge(vendedores[["id_vendedor", "nombre_vendedor"]], on="id_vendedor", validate="many_to_one")
          .merge(clientes[["id_cliente", "razon_social", "segmento"]], on="id_cliente", validate="many_to_one")
          .merge(productos[["sku", "producto", "categoria"]], on="sku", validate="many_to_one")
          .sort_values("id_linea"))
    return pd.DataFrame({"fecha": d.fecha.dt.date, "anio": d.fecha.dt.year, "mes": d.fecha.dt.month,
        "nombre_mes": d.fecha.dt.month.map(lambda m: MESES_ES[m - 1]), "trimestre": "T" + d.fecha.dt.quarter.astype(str),
        "id_factura": d.id_factura, "ciudad": d.ciudad, "region": d.region, "vendedor": d.nombre_vendedor,
        "cliente": d.razon_social, "segmento": d.segmento, "canal": d.canal, "categoria": d.categoria,
        "producto": d.producto, "cantidad": d.cantidad, "subtotal": d.subtotal, "margen_bruto": d.margen_bruto, "estado": d.estado})
def metas_planas(m):
    m = m.merge(sedes[["id_sede", "ciudad"]], on="id_sede")
    return pd.DataFrame({"ciudad": m.ciudad, "anio": m.anio, "mes": m.mes, "nombre_mes": m.mes.map(lambda x: MESES_ES[x - 1]),
        "fecha_mes": pd.to_datetime(dict(year=m.anio, month=m.mes, day=1)).dt.date, "meta_subtotal": m.meta_subtotal})

fuente = "Datos del curso"  # @param ["Datos del curso", "Mi Panel actualizado (subir)"]
if fuente.startswith("Mi Panel") and EN_COLAB:
    from google.colab import files
    archivo = list(files.upload())[0]
    datos = pd.read_excel(archivo, sheet_name="Ventas"); metas_p = pd.read_excel(archivo, sheet_name="Metas")
    datos["fecha"] = pd.to_datetime(datos["fecha"])
else:
    enero = pd.read_csv(ruta("nuevo_mes/ventas_2026_01.csv"), parse_dates=["fecha"])
    m_enero = pd.read_csv(ruta("nuevo_mes/metas_2026_01.csv"))
    datos = tabla_plana(pd.concat([ventas, enero])); datos["fecha"] = pd.to_datetime(datos["fecha"])
    metas_p = metas_planas(pd.concat([metas, m_enero]))
ok = datos[datos.estado != "Anulada"].copy()
CONTROL = 1204089720
ene26 = ok[(ok.anio == 2026) & (ok.mes == 1)]
display(Markdown(f"Leí **{miles(len(datos))} líneas**, de las cuales **{miles(len(datos[datos.anio == 2026]))}** son de enero de 2026."))
display(Markdown(f"**Cifra de control** · venta neta de enero de 2026 sin anuladas: **{pesos(ene26.subtotal.sum())}** "
                 + ("✓ coincide con la del tablero de Excel." if int(ene26.subtotal.sum()) == CONTROL else f"✗ no coincide con {pesos(CONTROL)}: revisa que pegaste todas las filas de enero.")))

## Paso 2 · Los cuatro gráficos del comité

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Cómo cerró enero? |
> | **Qué se grafica** | Tres indicadores: venta de enero, cumplimiento de la meta y variación contra enero de 2025 |
> | **Por qué esta forma** | Es lo primero que pregunta el gerente: tres cifras con su comparación. |
> | **Qué se descartó** | Una tabla con todas las sedes: responde después, no primero. |
> | **Cómo se lee** | El color de la variación dice si es buena o mala noticia. |
> | **En Excel 365** | Celdas grandes con IMPORTARDATOSDINAMICOS. |

In [ ]:
# @title Indicadores de enero
ene25 = ok[(ok.anio == 2025) & (ok.mes == 1)]
meta26 = metas_p[(metas_p.anio == 2026) & (metas_p.mes == 1)].meta_subtotal.sum()
v26, v25 = ene26.subtotal.sum(), ene25.subtotal.sum()
fig = go.Figure([
    go.Indicator(mode="number", value=v26 / 1e6, number=dict(prefix="$ ", suffix=" M", valueformat=",.0f"), title="Venta neta enero 2026", domain=dict(x=[0, 0.3])),
    go.Indicator(mode="number+delta", value=v26 / meta26 * 100, number=dict(suffix=" %", valueformat=".1f"), delta=dict(reference=100, valueformat=".1f", suffix=" pts"), title="Cumplimiento de la meta", domain=dict(x=[0.35, 0.65])),
    go.Indicator(mode="delta", value=v26 / 1e6, delta=dict(reference=v25 / 1e6, relative=True, valueformat=".1%", font=dict(size=56)), title="Variación contra enero 2025", domain=dict(x=[0.7, 1]))])
fig.update_layout(title=f"Enero cerró en {porc(v26 / meta26)} de la meta, {porc(v26 / v25 - 1, signo=True)} frente a enero de 2025", height=280)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué sede preocupa? |
> | **Qué se grafica** | Barras: cumplimiento de la meta de enero por sede, ordenado; línea punteada en 100 % |
> | **Por qué esta forma** | El cumplimiento es la pregunta; ordenarlo pone a la sede que preocupa en un extremo. |
> | **Qué se descartó** | Real y meta en pesos: Bogotá dominaría por tamaño. |
> | **Cómo se lee** | Las barras magenta no llegaron a la meta; la más corta es la que preocupa. |
> | **En Excel 365** | SUMAR.SI.CONJUNTO por sede contra tMetas. |

In [ ]:
# @title Cumplimiento de enero por sede
real = ene26.groupby("ciudad").subtotal.sum()
meta = metas_p[(metas_p.anio == 2026) & (metas_p.mes == 1)].groupby("ciudad").meta_subtotal.sum()
cum = (real / meta).sort_values()
fig = go.Figure(go.Bar(x=cum.values, y=cum.index, orientation="h", marker_color=[TEAL if v >= 1 else MAGENTA for v in cum.values],
                       text=[("✓ " if v >= 1 else "✗ ") + porc(v) for v in cum.values], textposition="outside", cliponaxis=False))
fig.add_vline(x=1, line_dash="dot", line_color="#11153A", annotation_text="Meta")
fig.update_layout(title=f"{cum.index[0]} es la sede que más preocupa: {porc(cum.iloc[0])} de su meta de enero", xaxis_tickformat=".0%",
                  xaxis_range=[0, max(1.2, cum.max() + 0.1)], height=360)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Es una caída puntual o una tendencia? |
> | **Qué se grafica** | Líneas: venta real y meta de los últimos 13 meses (enero 2025 a enero 2026) |
> | **Por qué esta forma** | Trece meses incluyen el mismo mes del año anterior; dos líneas en la misma unidad (pesos) se comparan sin trampas. |
> | **Qué se descartó** | Solo enero: no se sabe si es un mal mes o una tendencia. |
> | **Cómo se lee** | Si la línea real cae por debajo de la meta solo al final, es un mes atípico; si viene cayendo, es tendencia. |
> | **En Excel 365** | Gráfico de líneas sobre una dinámica con fecha agrupada por meses. |

In [ ]:
# @title Tendencia de 13 meses
ok["periodo"] = ok.fecha.dt.to_period("M").dt.to_timestamp()
r = ok[ok.periodo >= "2025-01-01"].groupby("periodo").subtotal.sum() / 1e6
m = metas_p.assign(periodo=pd.to_datetime(metas_p.fecha_mes)).query("periodo >= '2025-01-01'").groupby("periodo").meta_subtotal.sum() / 1e6
fig = go.Figure([go.Scatter(x=r.index, y=r.values, name="Real", mode="lines+markers", line=dict(color=AZUL, width=3)),
                 go.Scatter(x=m.index, y=m.values, name="Meta", mode="lines", line=dict(color=GRIS, dash="dash"))])
bajo = (r < m.reindex(r.index)).sum()
fig.update_layout(title=f"En {bajo} de los últimos 13 meses la venta quedó por debajo de la meta", yaxis_title=MM, legend_title="", hovermode="x unified")
eje_meses(fig, r.index)
fig.show()

> **Ficha de decisión**
>
> | | |
> |---|---|
> | **Pregunta** | ¿Qué está pasando con los canales? |
> | **Qué se grafica** | Barras apiladas al 100 %: reparto por canal en enero de 2025 y enero de 2026 |
> | **Por qué esta forma** | Compara la composición del mismo mes en dos años, sin el efecto del tamaño. |
> | **Qué se descartó** | Valores absolutos: la caída general taparía el cambio de mezcla. |
> | **Cómo se lee** | Compara el ancho de cada color entre las dos barras. |
> | **En Excel 365** | Dinámica con canal en columnas y Mostrar valores como › % del total de fila. |

In [ ]:
# @title Participación por canal, enero contra enero
ce = ok[(ok.mes == 1) & ok.anio.isin([2025, 2026])]
p = pd.crosstab(ce.anio, ce.canal, values=ce.subtotal, aggfunc="sum", normalize="index")[["Presencial", "Televentas", "E-commerce"]]
fig = go.Figure([go.Bar(name=c, y=["Enero " + str(a) for a in p.index], x=p[c], orientation="h", marker_color=col, text=[f"{v:.0%}" for v in p[c]],
                        marker_line_color="white", marker_line_width=2) for c, col in zip(p.columns, [AZUL, TEAL, MAGENTA])])
fig.update_layout(barmode="stack", xaxis_tickformat=".0%", yaxis=dict(autorange="reversed"), height=300, legend_title="",
                  title=f"El e-commerce pasó de {porc(p.iloc[0, 2])} a {porc(p.iloc[1, 2])} de la venta de enero")
fig.show()

## Paso 3 · Generar el tablero web del comité
Un solo archivo HTML con una lista para elegir la sede. Se abre en cualquier navegador, sin Colab ni internet.

In [ ]:
# @title Crear y descargar tablero_comite_enero2026.html
def figuras(c):
    d = ok if c == "Todas" else ok[ok.ciudad == c]
    mt = metas_p if c == "Todas" else metas_p[metas_p.ciudad == c]
    e26, e25 = d[(d.anio == 2026) & (d.mes == 1)].subtotal.sum(), d[(d.anio == 2025) & (d.mes == 1)].subtotal.sum()
    me = mt[(mt.anio == 2026) & (mt.mes == 1)].meta_subtotal.sum()
    k = go.Figure([go.Indicator(mode="number", value=e26 / 1e6, number=dict(prefix="$ ", suffix=" M", valueformat=",.0f"), title="Venta enero 2026", domain=dict(x=[0, 0.3])),
                   go.Indicator(mode="number+delta", value=e26 / me * 100, number=dict(suffix=" %", valueformat=".1f"), delta=dict(reference=100, valueformat=".1f", suffix=" pts"), title="Cumplimiento", domain=dict(x=[0.35, 0.65])),
                   go.Indicator(mode="delta", value=e26 / 1e6, delta=dict(reference=e25 / 1e6, relative=True, valueformat=".1%", font=dict(size=56)), title="Variación contra enero 2025", domain=dict(x=[0.7, 1]))])
    k.update_layout(height=230, margin=dict(t=40, b=10))
    rr = d[d.periodo >= "2025-01-01"].groupby("periodo").subtotal.sum() / 1e6
    mm = mt.assign(periodo=pd.to_datetime(mt.fecha_mes)).query("periodo >= '2025-01-01'").groupby("periodo").meta_subtotal.sum() / 1e6
    t = go.Figure([go.Scatter(x=rr.index, y=rr.values, name="Real", mode="lines+markers", line=dict(color=AZUL, width=3)),
                   go.Scatter(x=mm.index, y=mm.values, name="Meta", mode="lines", line=dict(color=GRIS, dash="dash"))])
    t.update_layout(title="Real contra meta, últimos 13 meses (millones)", height=340, hovermode="x unified")
    eje_meses(t, rr.index)
    cc = d[(d.mes == 1) & d.anio.isin([2025, 2026])]
    pp = pd.crosstab(cc.anio, cc.canal, values=cc.subtotal, aggfunc="sum", normalize="index").reindex(columns=["Presencial", "Televentas", "E-commerce"]).fillna(0)
    ch = go.Figure([go.Bar(name=x, y=["Ene " + str(a) for a in pp.index], x=pp[x], orientation="h", marker_color=col, text=[f"{v:.0%}" for v in pp[x]]) for x, col in zip(pp.columns, [AZUL, TEAL, MAGENTA])])
    ch.update_layout(barmode="stack", title="Participación por canal", xaxis_tickformat=".0%", height=260, yaxis=dict(autorange="reversed"))
    return [k, t, ch]
opciones = ["Todas", *ORDEN_CIUDADES]
cab = f"""<html><head><meta charset="utf-8"><title>Comité comercial · enero 2026</title><style>
body{{font-family:Segoe UI,Arial,sans-serif;max-width:1100px;margin:20px auto;padding:0 16px;color:#11153A}} h1{{color:#00006A;margin:0}}
.meta{{color:#566080}} select{{font-size:16px;padding:6px 10px;margin:14px 0}} .vista{{display:none}} .vista.on{{display:block}}</style></head><body>
<h1>Comité comercial · enero 2026</h1><div class="meta">Distribuidora Andina S.A.S. · venta neta sin anuladas · fuente: ERP con corte al 31/01/2026</div>"""
# cumplimiento de todas las sedes (siempre visible)
cumf = go.Figure(go.Bar(x=cum.values, y=cum.index, orientation="h", marker_color=[TEAL if v >= 1 else MAGENTA for v in cum.values], text=[porc(v) for v in cum.values], textposition="outside"))
cumf.add_vline(x=1, line_dash="dot"); cumf.update_layout(title="Cumplimiento de la meta de enero por sede", xaxis_tickformat=".0%", height=320, xaxis_range=[0, max(1.2, cum.max() + 0.1)])
html = [cab, cumf.to_html(full_html=False, include_plotlyjs=True), '<label>Ver sede: <select id="s">' + "".join(f"<option>{o}</option>" for o in opciones) + "</select></label>"]
for i, o in enumerate(opciones):
    html.append(f'<div class="vista{" on" if i == 0 else ""}" data-o="{o}">' + "".join(f.to_html(full_html=False, include_plotlyjs=False) for f in figuras(o)) + "</div>")
html.append("""<script>document.getElementById('s').onchange=function(){document.querySelectorAll('.vista').forEach(function(v){v.classList.toggle('on',v.dataset.o===this.value);if(v.dataset.o===this.value){v.querySelectorAll('.plotly-graph-div').forEach(function(g){Plotly.Plots.resize(g)})}},this)};</script>
<p class="meta">Datos ficticios · Excel Corporativo · 12 Claves · Dr. Bits</p></body></html>""")
with open("tablero_comite_enero2026.html", "w", encoding="utf-8") as fh:
    fh.write("".join(html))

if EN_COLAB:
    from google.colab import files
    files.download("tablero_comite_enero2026.html")
    display(Markdown("Descarga iniciada: **tablero_comite_enero2026.html**. Si no aparece, búscalo en el panel de archivos (ícono de carpeta) › clic derecho › Descargar."))
else:
    display(Markdown("Archivo creado: **tablero_comite_enero2026.html** (en la carpeta del cuaderno)."))

## Paso 4 · Borrador del informe
Un punto de partida con las cifras de este cuaderno. Reescríbelo con tus palabras y agrega lo que viste en el Panel de Excel.

In [ ]:
# @title Generar el borrador
peor, mejor = cum.index[0], cum.index[-1]
display(Markdown(f"""
**INFORME COMERCIAL · ENERO 2026**

1. Enero cerró en **{millones(v26)}**, el **{porc(v26 / meta26)}** de la meta y **{porc(v26 / v25 - 1, signo=True)}** frente a enero de 2025. *Siguiente pregunta:* ¿la caída es de volumen (menos facturas) o de valor (facturas más pequeñas)?
2. **{peor}** llegó al **{porc(cum.iloc[0])}** de su meta y **{mejor}** al **{porc(cum.iloc[-1])}**. *Siguiente pregunta:* ¿qué clientes grandes de {peor} no compraron en enero?
3. El e-commerce pasó de **{porc(p.iloc[0, 2])}** a **{porc(p.iloc[1, 2])}** de la venta de enero. *Siguiente pregunta:* ¿el canal digital está reemplazando ventas presenciales o sumando clientes nuevos?

**Recomendación:** revisar con el coordinador de {peor} la cartera de clientes principales antes del 15 de febrero.
"""))

## Resumen
* Un proceso bien armado se repite cada mes: pegar, actualizar, verificar la cifra de control.
* Excel y Colab deben dar la misma cifra; si no, algo se perdió en el camino.
* El comité necesita tres respuestas con cifra, comparación y siguiente pregunta, no veinte gráficos.

**Siguiente:** [Volver al laboratorio](https://omarli-360.github.io/excel-corporativo-12-claves/#laboratorio)

<sub>Dr. Bits · ADNBITS S.A.S. · Contenido CC BY-SA 4.0 · Datos ficticios de Distribuidora Andina S.A.S.</sub>